# FoodSave · Modelo de ventas diarias con CatBoost
Sube este notebook a Google Colab y ejecuta las celdas en orden. En la carga se solicitará tu CSV con **date, article, Quantity**.

Con el CSV legacy de panadería: entrenamiento antes de abril de 2022, validación abril–junio y test julio–septiembre de 2022. Con un CSV normalizado de otro comercio, se reservan por defecto sus 30 últimas fechas registradas para test y las 30 anteriores para validación; las previas son entrenamiento. CatBoost usa la validación para detener el entrenamiento. Como producir de menos es más costoso en este piloto, utiliza una pérdida cuantil con `alpha=0.65` seleccionada en validaciones temporales anteriores al test. El test se consulta después de entrenar y no modifica el modelo.

Los historiales desconocidos permanecen como valores faltantes y CatBoost puede tratarlos. Se requiere un mínimo configurable de observaciones previas; ya no se descarta un mes entero por un solo día sin transacciones. Las métricas nuevas no son directamente comparables con las anteriores porque ahora se evalúan más filas.

Este es un backtest de **un día adelante**: para cada fecha se conocen las ventas reales hasta el día anterior, incluso dentro del test. No pronostica tres meses desde junio.

Las ventas observadas pueden estar limitadas por stock. Las diferencias entre predicción y ventas no miden por sí solas demanda no atendida ni desperdicio real.


In [ ]:
%pip install -q "catboost>=1.2,<2" "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4"

## 1. Configuración
CSV legacy (`date`, `article`, `Quantity`) o normalizado (`comercio_id`, `sucursal_id`, `fecha_local`, `producto_id`, `unidades_vendidas`). Si el normalizado incluye varias sucursales, elige una con `COMERCIO_ID` y `SUCURSAL_ID`. En el formato normalizado la división temporal es automática; puedes fijarla manualmente con `SPLIT_AUTOMATICO=False` y las tres fechas de configuración. Fechas ISO por defecto. Cambia `FORMATO_FECHA` si usa otro formato (por ejemplo `"%d/%m/%Y"`) y `SEPARADOR` si usa punto y coma.

**Ausencias:** por defecto se consideran desconocidas. El objetivo del día debe ser conocido; los historiales numéricos pueden contener valores faltantes y CatBoost los procesa. Si confirmas que en días con transacciones un producto ausente significa cero ventas, activa `CERO_EN_DIAS_CON_REGISTROS`. Los días sin ninguna transacción siguen desconocidos, salvo que confirmes `CERO_EN_DIAS_SIN_REGISTROS`. No se imputan ceros antes de la primera aparición del producto. Revisa altas, bajas y cobertura del catálogo.

Las cantidades negativas se excluyen por defecto para modelar unidades vendidas positivas. Si representan devoluciones, esta decisión produce ventas brutas, no ventas netas. El notebook informa cuántas filas se excluyeron y conserva el CSV original. Cambia `NEGATIVOS` a `"error"` si quieres detenerte para revisarlas. No se eliminan duplicados automáticamente: pueden ser transacciones legítimas.

In [ ]:
from pathlib import Path
import json, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

CSV_PATH = ""  # En Colab, vacío abre el selector. Localmente indica una ruta.
COMERCIO_ID = ""  # Si el CSV normalizado tiene varias sucursales, indica el comercio.
SUCURSAL_ID = ""  # Si el CSV normalizado tiene varias sucursales, indica la sucursal.
SPLIT_AUTOMATICO = None  # None: automático para CSV normalizado; fijo para legacy
FECHAS_VALIDACION = 30  # fechas con registros de la sucursal
FECHAS_TEST = 30  # fechas con registros de la sucursal
SEPARADOR = ","
FORMATO_FECHA = "%Y-%m-%d"
CERO_EN_DIAS_CON_REGISTROS = False
CERO_EN_DIAS_SIN_REGISTROS = False
NEGATIVOS = "excluir"  # "excluir" (ventas brutas) o "error"
MIN_OBSERVACIONES_28 = 7  # mínimo de fechas conocidas del producto en los 28 días previos
CUANTIL = 0.65  # provisional: prioriza reducir unidades faltantes
COSTO_RELATIVO_FALTANTE = 2.0  # por cada unidad faltante
COSTO_RELATIVO_SOBRANTE = 1.0  # por cada unidad sobrante
INICIO_VALIDACION = pd.Timestamp("2022-04-01")
INICIO_TEST = pd.Timestamp("2022-07-01")
FIN_TEST_EXCLUSIVO = pd.Timestamp("2022-10-01")
SALIDA = Path("foodsave_resultados")
SALIDA.mkdir(exist_ok=True)

In [ ]:
if not CSV_PATH:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Fuera de Colab configura CSV_PATH.") from exc
    subidos = files.upload()
    candidatos = [nombre for nombre in subidos if nombre.lower().endswith(".csv")]
    if len(candidatos) != 1:
        raise ValueError("Sube exactamente un archivo CSV.")
    CSV_PATH = candidatos[0]

original = pd.read_csv(CSV_PATH, sep=SEPARADOR)
df = original.copy()  # El archivo original no se modifica.
df.columns = df.columns.str.strip()
columnas_estandar = {
    "comercio_id", "sucursal_id", "fecha_local", "producto_id", "unidades_vendidas"
}
FORMATO_ORIGEN = "estandar" if columnas_estandar.issubset(df.columns) else "bakery"
comercio_elegido = None
sucursal_elegida = None
if FORMATO_ORIGEN == "estandar":
    df["comercio_id"] = df["comercio_id"].astype("string").str.strip()
    df["sucursal_id"] = df["sucursal_id"].astype("string").str.strip()
    tiendas = df[["comercio_id", "sucursal_id"]].drop_duplicates()
    if len(tiendas) == 1 and not COMERCIO_ID and not SUCURSAL_ID:
        comercio_elegido = str(tiendas.iloc[0]["comercio_id"])
        sucursal_elegida = str(tiendas.iloc[0]["sucursal_id"])
    elif COMERCIO_ID and SUCURSAL_ID:
        comercio_elegido, sucursal_elegida = COMERCIO_ID, SUCURSAL_ID
    else:
        display(tiendas)
        raise ValueError("Indica COMERCIO_ID y SUCURSAL_ID para elegir una sucursal.")
    df = df.loc[
        df.comercio_id.eq(comercio_elegido) & df.sucursal_id.eq(sucursal_elegida)
    ].copy()
    if df.empty:
        raise ValueError("No hay filas para el comercio y la sucursal seleccionados.")
    df = df.rename(columns={
        "fecha_local": "date", "producto_id": "article",
        "unidades_vendidas": "Quantity",
    })
    print(f"Sucursal seleccionada: {comercio_elegido} / {sucursal_elegida}")
requeridas = {"date", "article", "Quantity"}
if not requeridas.issubset(df.columns):
    raise ValueError(f"Faltan columnas: {requeridas - set(df.columns)}")
df["date"] = pd.to_datetime(df["date"], format=FORMATO_FECHA, errors="coerce").dt.normalize()
df["article"] = df["article"].astype("string").str.strip()
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
invalidas = df["date"].isna() | df["article"].isna() | df["article"].eq("") | ~np.isfinite(df["Quantity"])
if invalidas.any():
    display(df.loc[invalidas].head(10))
    raise ValueError(f"Hay {invalidas.sum()} filas inválidas; corrige el CSV antes de entrenar.")
if NEGATIVOS not in {"error", "excluir"}:
    raise ValueError("NEGATIVOS debe ser error o excluir.")
negativas = df["Quantity"] < 0
filas_originales = len(df)
cantidad_negativas = int(negativas.sum())
print(f"Filas: {filas_originales:,}; cantidades negativas: {cantidad_negativas:,} ({cantidad_negativas / filas_originales:.2%})")
if negativas.any() and NEGATIVOS == "error":
    raise ValueError("Revisa devoluciones o configura NEGATIVOS='excluir'.")
df = df.loc[~negativas].copy()
if cantidad_negativas:
    print(f"Se excluyeron {cantidad_negativas:,} filas con Quantity negativa; quedan {len(df):,} para modelar ventas brutas.")
if df.empty:
    raise ValueError("No quedan ventas válidas.")
modo_automatico = (FORMATO_ORIGEN == "estandar") if SPLIT_AUTOMATICO is None else SPLIT_AUTOMATICO
if modo_automatico:
    fechas_registradas = pd.DatetimeIndex(sorted(df.date.unique()))
    if df.loc[df.Quantity > 0, "date"].nunique() < 180:
        raise ValueError(
            "Para el piloto automático se requieren al menos 180 fechas con ventas de la sucursal."
        )
    if FECHAS_VALIDACION < 1 or FECHAS_TEST < 1:
        raise ValueError("FECHAS_VALIDACION y FECHAS_TEST deben ser positivas.")
    if len(fechas_registradas) <= FECHAS_VALIDACION + FECHAS_TEST + 28:
        raise ValueError("No quedan suficientes fechas previas para entrenar.")
    INICIO_TEST = pd.Timestamp(fechas_registradas[-FECHAS_TEST])
    INICIO_VALIDACION = pd.Timestamp(
        fechas_registradas[-(FECHAS_VALIDACION + FECHAS_TEST)]
    )
    FIN_TEST_EXCLUSIVO = pd.Timestamp(fechas_registradas[-1]) + pd.Timedelta(days=1)
    print(
        "División temporal automática:",
        "train antes de", INICIO_VALIDACION.date(),
        "validación hasta", (INICIO_TEST - pd.Timedelta(days=1)).date(),
        "test desde", INICIO_TEST.date(), "hasta", (FIN_TEST_EXCLUSIVO - pd.Timedelta(days=1)).date(),
    )
print(f"Fechas: {df.date.min().date()} a {df.date.max().date()}; productos: {df.article.nunique()}")
display(df.head())
display(df[["date", "article", "Quantity"]].isna().sum())

## 2. Ventas por día y cobertura
El calendario completo hace que un lag de siete filas represente exactamente siete días, aunque falten registros. Los valores desconocidos permanecen como NaN. Las medias históricas utilizan solo días con ventas conocidas; su conteo indica cuántas observaciones sustentan cada promedio. Un hueco del calendario no se convierte en cero ni descarta automáticamente los 28 días siguientes.


In [ ]:
agregadas = df.groupby(["date", "article"], as_index=False)["Quantity"].sum().rename(columns={"Quantity": "ventas_dia"})
fechas = pd.date_range(agregadas.date.min(), agregadas.date.max(), freq="D")
productos = sorted(agregadas.article.unique())
indice = pd.MultiIndex.from_product([fechas, productos], names=["date", "article"])
diarias = agregadas.set_index(["date", "article"]).reindex(indice).reset_index()
primera = agregadas.groupby("article")["date"].min()
activo = diarias.date >= diarias.article.map(primera)
dia_con_registros = diarias.date.isin(agregadas.date.unique())
imputable = activo & (
    (dia_con_registros & CERO_EN_DIAS_CON_REGISTROS)
    | (~dia_con_registros & CERO_EN_DIAS_SIN_REGISTROS)
)
diarias["cero_imputado"] = diarias.ventas_dia.isna() & imputable
diarias.loc[diarias.cero_imputado, "ventas_dia"] = 0.0
print("Días sin ninguna transacción:", len(fechas.difference(agregadas.date.unique())))
print("Ceros imputados:", int(diarias.cero_imputado.sum()))
print("Objetivos desconocidos:", int(diarias.ventas_dia.isna().sum()))
display(agregadas.groupby("article").agg(dias_registrados=("date", "size"), unidades=("ventas_dia", "sum")))
diarias.to_csv(SALIDA / "ventas_diarias.csv", index=False)

In [ ]:
FEATURES = [
    "article", "dia_semana", "mes", "dia_mes", "fin_semana",
    "ventas_ayer", "ventas_hace_7_dias",
    "promedio_7_dias", "promedio_14_dias", "promedio_28_dias",
    "conteo_7_dias", "conteo_14_dias", "conteo_28_dias",
]

def crear_features(tabla):
    salida = tabla.sort_values(["article", "date"]).reset_index(drop=True).copy()
    salida["dia_semana"] = salida.date.dt.dayofweek
    salida["mes"] = salida.date.dt.month
    salida["dia_mes"] = salida.date.dt.day
    salida["fin_semana"] = (salida.dia_semana >= 5).astype(int)
    grupos = salida.groupby("article")["ventas_dia"]
    salida["ventas_ayer"] = grupos.shift(1)
    salida["ventas_hace_7_dias"] = grupos.shift(7)
    for ventana in (7, 14, 28):
        salida[f"promedio_{ventana}_dias"] = grupos.transform(
            lambda x: x.shift(1).rolling(ventana, min_periods=1).mean()
        )
        salida[f"conteo_{ventana}_dias"] = grupos.transform(
            lambda x: x.shift(1).rolling(ventana, min_periods=1).count()
        )
    return salida

con_features = crear_features(diarias)
# Solo el objetivo debe estar observado. Los lags faltantes son información,
# no ceros; CatBoost admite NaN en variables numéricas.
datos = con_features.loc[
    con_features.ventas_dia.notna()
    & (con_features.conteo_28_dias >= MIN_OBSERVACIONES_28)
].copy()
train = datos.loc[datos.date < INICIO_VALIDACION].copy()
validation = datos.loc[
    (datos.date >= INICIO_VALIDACION) & (datos.date < INICIO_TEST)
].copy()
for nombre, parte in [("train", train), ("validation", validation)]:
    if parte.empty:
        raise ValueError(
            f"{nombre} vacío: revisa fechas y el mínimo de observaciones previas."
        )
    print(
        nombre, "filas:", len(parte), "fechas:", parte.date.nunique(),
        "primera:", parte.date.min().date(), "última:", parte.date.max().date(),
        "productos:", parte.article.nunique(),
    )
assert train.date.max() < validation.date.min() < INICIO_TEST
print(f"Cobertura total con objetivo e historial mínimo: {len(datos) / len(diarias):.1%}")

def cobertura_periodo(parte, inicio, fin):
    observadas = agregadas.loc[
        (agregadas.date >= inicio) & (agregadas.date < fin)
    ]
    return {
        "filas_evaluables": len(parte),
        "filas_con_ventas": len(observadas),
        "cobertura_filas_pct": 100 * len(parte) / len(observadas) if len(observadas) else np.nan,
        "unidades_evaluables": parte.ventas_dia.sum(),
        "unidades_con_ventas": observadas.ventas_dia.sum(),
        "cobertura_unidades_pct": (
            100 * parte.ventas_dia.sum() / observadas.ventas_dia.sum()
            if observadas.ventas_dia.sum() else np.nan
        ),
    }

display(pd.DataFrame([
    {"periodo": "entrenamiento", **cobertura_periodo(train, diarias.date.min(), INICIO_VALIDACION)},
    {"periodo": "validacion", **cobertura_periodo(validation, INICIO_VALIDACION, INICIO_TEST)},
]))


## 3. Entrenar CatBoost con prioridad de faltantes
CatBoost recibe las variables de calendario e historial. `CUANTIL=0.65` busca reducir faltantes; puede aumentar excedentes y WAPE. La comparación de costo usa pesos provisionales de 2 para una unidad faltante y 1 para una sobrante. Usa `article` directamente como categoría. Aprende con las filas anteriores a abril de 2022 y observa abril–junio para detener el entrenamiento si el error de validación deja de mejorar. Los lags desconocidos permanecen como NaN; los conteos históricos indican cuánto respaldo tienen los promedios. CatBoost admite NaN numéricos. Las predicciones negativas se limitan a cero para calcular métricas; aún no se redondean.


In [ ]:
modelo = CatBoostRegressor(
    iterations=500,
    depth=7,
    learning_rate=0.05,
    loss_function=f"Quantile:alpha={CUANTIL}",
    nan_mode="Min",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)
modelo.fit(
    train[FEATURES],
    train["ventas_dia"],
    cat_features=["article"],
    eval_set=(validation[FEATURES], validation["ventas_dia"]),
    early_stopping_rounds=50,
    use_best_model=True,
)
print("Árboles conservados:", modelo.tree_count_)

def predecir(filas):
    return np.maximum(0, modelo.predict(filas[FEATURES]))


In [ ]:
def metricas(real, pred):
    real, pred = np.asarray(real, dtype=float), np.asarray(pred, dtype=float)
    error = np.abs(real - pred)
    positivos = real > 0
    relativo = error / np.maximum(real, 1)
    faltante = np.maximum(real - pred, 0)
    sobrante = np.maximum(pred - real, 0)
    return {
        "n": len(real), "MAE": mean_absolute_error(real, pred),
        "unidades_faltantes": faltante.sum(),
        "unidades_sobrantes": sobrante.sum(),
        "sesgo_unidades": pred.sum() - real.sum(),
        "costo_relativo": (
            COSTO_RELATIVO_FALTANTE * faltante.sum()
            + COSTO_RELATIVO_SOBRANTE * sobrante.sum()
        ),
        "WAPE_pct": 100 * error.sum() / real.sum() if real.sum() > 0 else np.nan,
        "±10%": 100 * (relativo <= .10).mean(),
        "±20%": 100 * (relativo <= .20).mean(),
        "n_ventas_cero": int((~positivos).sum()),
        "MAE_ventas_cero": error[~positivos].mean() if (~positivos).any() else np.nan,
    }


predicciones_validacion = validation[["date", "article", "ventas_dia"]].copy()
predicciones_validacion["prediccion"] = predecir(validation)
metricas_validacion = pd.DataFrame([{
    "Modelo": "CatBoost",
    **metricas(predicciones_validacion.ventas_dia, predicciones_validacion.prediccion),
}])
display(metricas_validacion)


Como el piloto prioriza evitar faltantes, revisa `unidades_faltantes`, `unidades_sobrantes`, `sesgo_unidades` y `costo_relativo` junto a las métricas tradicionales. Estos pesos son supuestos del piloto, no costos monetarios medidos. MAE es el error medio en unidades por producto y día; WAPE compara el error total con las ventas totales. Los porcentajes ±10% y ±20% usan error absoluto / max(ventas reales, 1), como en la guía. En ventas cero ese cociente no es un porcentaje relativo convencional; por eso también se muestra el MAE de esas filas. WAPE es indefinido si las ventas totales son cero.

La validación permite revisar el modelo y detener su entrenamiento. El test siguiente mide el rendimiento en fechas posteriores; no se usa para reajustar el modelo.


## 4. Test final de CatBoost
Se evalúa el modelo ya entrenado en julio–septiembre de 2022. El simulador utiliza este mismo modelo.


In [ ]:
test = datos.loc[(datos.date >= INICIO_TEST) & (datos.date < FIN_TEST_EXCLUSIVO)].copy()
if test.empty:
    raise ValueError("Test vacío: revisa fechas y disponibilidad de 28 días de historial.")
assert train.date.max() < validation.date.min() < test.date.min()
predicciones_test = test[["date", "article", "ventas_dia"]].copy()
predicciones_test["prediccion"] = predecir(test)
metricas_test = pd.DataFrame([{"Modelo": "CatBoost", **metricas(test.ventas_dia, predicciones_test.prediccion)}])
display(metricas_test)
metricas_producto = pd.DataFrame([
    {"article": producto, **metricas(grupo.ventas_dia, grupo.prediccion)}
    for producto, grupo in predicciones_test.groupby("article")
])
dias_train = train.groupby("article").size()
dias_validacion = validation.groupby("article").size()
metricas_producto["dias_train"] = metricas_producto.article.map(dias_train).fillna(0).astype(int)
metricas_producto["dias_validacion"] = metricas_producto.article.map(dias_validacion).fillna(0).astype(int)
metricas_producto["muestra_suficiente"] = (
    (metricas_producto.dias_train >= 90)
    & (metricas_producto.dias_validacion >= 20)
    & (metricas_producto.n >= 20)
)
print("Productos con muestra suficiente para reportar métrica individual:",
      int(metricas_producto.muestra_suficiente.sum()), "/", len(metricas_producto))
display(metricas_producto)
potenciales = diarias.loc[(diarias.date >= INICIO_TEST) & (diarias.date < FIN_TEST_EXCLUSIVO)]
print(f"Cobertura test en el calendario: {len(test)} / {len(potenciales)} combinaciones ({len(test)/len(potenciales):.1%})")
display(pd.DataFrame([{
    "periodo": "test",
    **cobertura_periodo(test, INICIO_TEST, FIN_TEST_EXCLUSIVO),
}]))
display(test.groupby(test.date.dt.to_period("M")).agg(
    filas=("article", "size"), fechas=("date", "nunique"),
    productos=("article", "nunique"), unidades=("ventas_dia", "sum"),
))
print("Última fecha evaluable del test:", test.date.max().date())
print("Productos del test no vistos en train:", sorted(set(test.article) - set(train.article)))
ax = predicciones_test.groupby("date")[["ventas_dia", "prediccion"]].sum().plot(
    figsize=(14, 4), title="Test final: CatBoost (solo filas evaluables)"
)
ax.set_ylabel("Unidades")
plt.show()


## 5. Simulador histórico
Cambia la fecha y ejecuta. El simulador elige productos solo por su historial anterior (al menos `MIN_OBSERVACIONES_28` fechas conocidas en los 28 días previos), sin mirar si se vendieron ese día. Así también puede mostrar predicciones para una fecha sin transacciones. Con `MOSTRAR_REALES=True`, una venta ausente se mostrará como **desconocida**, no como cero. El MAE diario utiliza únicamente las ventas conocidas.


In [ ]:
def filas_dia(fecha):
    try:
        fecha = pd.to_datetime(fecha, format="%Y-%m-%d", errors="raise").normalize()
    except (ValueError, TypeError) as exc:
        raise ValueError("Usa una fecha válida YYYY-MM-DD.") from exc
    if pd.isna(fecha) or not INICIO_TEST <= fecha < FIN_TEST_EXCLUSIVO or fecha > diarias.date.max():
        raise ValueError(f"Selecciona una fecha de test entre {INICIO_TEST.date()} y {(FIN_TEST_EXCLUSIVO - pd.Timedelta(days=1)).date()}.")
    # La lista de productos depende exclusivamente del historial anterior.
    filas = con_features.loc[
        (con_features.date == fecha)
        & (con_features.conteo_28_dias >= MIN_OBSERVACIONES_28)
    ].copy()
    if filas.empty:
        raise ValueError("La fecha no tiene productos con suficiente historial previo.")
    return filas

def predecir_dia(fecha):
    filas = filas_dia(fecha)
    return pd.DataFrame({
        "article": filas.article.to_numpy(),
        "prediccion": np.rint(predecir(filas)).astype(int),
    }).sort_values("prediccion", ascending=False).reset_index(drop=True)

def comparar_dia(fecha):
    filas = filas_dia(fecha)
    resultado = predecir_dia(fecha).merge(
        filas[["article", "ventas_dia"]], on="article", validate="one_to_one"
    )
    resultado["error"] = (resultado.prediccion - resultado.ventas_dia).abs()
    resultado["error_pct"] = 100 * resultado.error / np.maximum(resultado.ventas_dia, 1)
    resultado["exceso_simulado"] = (resultado.prediccion - resultado.ventas_dia).clip(lower=0)
    resultado["deficit_simulado"] = (resultado.ventas_dia - resultado.prediccion).clip(lower=0)
    return resultado.sort_values("ventas_dia", ascending=False).reset_index(drop=True)

def simulador_foodsave(fecha, mostrar_reales=False):
    pred = predecir_dia(fecha)
    print(f"FoodSave · CatBoost · {fecha} · {len(pred)} de {len(productos)} productos con historial suficiente")
    display(pred)
    if mostrar_reales:
        comparacion = comparar_dia(fecha)
        display(comparacion)
        conocidos = comparacion.error.notna()
        print(f"Ventas conocidas: {int(conocidos.sum())} / {len(comparacion)} productos pronosticados")
        if conocidos.any():
            print(f"MAE del día sobre ventas conocidas (predicción redondeada): {comparacion.loc[conocidos, 'error'].mean():.2f} unidades")
        else:
            print("No hay ventas registradas para comparar; las ausencias permanecen desconocidas.")
    return pred


In [ ]:
FECHA_PRUEBA = str(test.date.min().date())  # Por ejemplo: "2022-08-24"
MOSTRAR_REALES = False
simulador_foodsave(FECHA_PRUEBA, mostrar_reales=MOSTRAR_REALES);

## 6. Guardar y descargar
Se guarda el modelo CatBoost entrenado, sus métricas y metadatos. El ZIP es experimental; descárgalo antes de cerrar Colab, ya que su almacenamiento es temporal.


In [ ]:
artefacto = "catboost_model.cbm"
modelo.save_model(str(SALIDA / artefacto))
recargado = CatBoostRegressor()
recargado.load_model(str(SALIDA / artefacto))
np.testing.assert_allclose(
    modelo.predict(test[FEATURES].head()),
    recargado.predict(test[FEATURES].head()),
)
metricas_validacion.to_csv(SALIDA / "metricas_validacion.csv", index=False)
predicciones_validacion.to_csv(SALIDA / "predicciones_validacion.csv", index=False)
metricas_test.to_csv(SALIDA / "metricas_test.csv", index=False)
metricas_producto.to_csv(SALIDA / "metricas_test_por_producto.csv", index=False)
predicciones_test.to_csv(SALIDA / "predicciones_test.csv", index=False)
metadata = {
    "estado": "experimental_pendiente_de_aceptacion",
    "formato_origen": FORMATO_ORIGEN,
    "split_automatico": bool(modo_automatico),
    "fechas_validacion_configuradas": FECHAS_VALIDACION if modo_automatico else None,
    "fechas_test_configuradas": FECHAS_TEST if modo_automatico else None,
    "comercio_id": comercio_elegido,
    "sucursal_id": sucursal_elegida,
    "modelo": "CatBoost",
    "loss_function": f"Quantile:alpha={CUANTIL}",
    "costo_relativo_faltante": COSTO_RELATIVO_FALTANTE,
    "costo_relativo_sobrante": COSTO_RELATIVO_SOBRANTE,
    "artefacto": artefacto,
    "features": FEATURES,
    "target": "ventas_dia",
    "horizonte": "un_dia_con_historial_real",
    "inicio_validacion": str(INICIO_VALIDACION.date()),
    "inicio_test": str(INICIO_TEST.date()),
    "fin_test_exclusivo": str(FIN_TEST_EXCLUSIVO.date()),
    "min_observaciones_previas_28_dias": MIN_OBSERVACIONES_28,
    "cero_en_dias_con_registros": CERO_EN_DIAS_CON_REGISTROS,
    "cero_en_dias_sin_registros": CERO_EN_DIAS_SIN_REGISTROS,
    "negativos": NEGATIVOS,
    "filas_originales": filas_originales,
    "filas_negativas_excluidas": cantidad_negativas,
    "filas_utilizadas": len(df),
    "formato_fecha": FORMATO_FECHA,
    "productos_entrenados": sorted(train.article.unique().tolist()),
    "filas_evaluables": len(datos),
    "filas_calendario": len(diarias),
    "seed": 42,
    "arboles_conservados": modelo.tree_count_,
    "python": platform.python_version(),
    "versiones": {
        p: importlib.metadata.version(p)
        for p in ["catboost", "pandas", "numpy", "scikit-learn", "matplotlib"]
    },
}
(SALIDA / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
import zipfile
archivo_zip = str(Path("foodsave_resultados.zip").resolve())
archivos_exportados = [
    artefacto, "metadata.json", "ventas_diarias.csv",
    "metricas_validacion.csv", "predicciones_validacion.csv",
    "metricas_test.csv", "metricas_test_por_producto.csv", "predicciones_test.csv",
]
with zipfile.ZipFile(archivo_zip, "w", compression=zipfile.ZIP_DEFLATED) as paquete:
    for nombre_archivo in archivos_exportados:
        paquete.write(SALIDA / nombre_archivo, arcname=nombre_archivo)
print("Resultados guardados:", archivo_zip)


In [ ]:
DESCARGAR_ZIP = True
if DESCARGAR_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print("Ejecución local: abre", archivo_zip)
    else:
        files.download(archivo_zip)